# Thesis Analysis: The Gold Standard Dataset
This notebook synthesizes the final experimental dataset, isolating the relationship between swarm density ($N$) and collective transport efficiency, as well as providing a control comparison against convex geometries (The Torque Debt).


In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

all_trials = []
base_log_dir = 'logs'

for root, dirs, files in os.walk(base_log_dir):
    for file in files:
        if file.endswith('.json'):
            folder_name = os.path.basename(root)
            file_path = os.path.join(root, file)
            
            # Determine shape and N from folder name
            if folder_name.startswith('N_'):
                shape = 'l_shape'
                n_value = int(folder_name.split('_')[1])
            elif folder_name.startswith('square_N_'):
                shape = 'square'
                n_value = int(folder_name.split('_')[2])
            else:
                continue
                
            with open(file_path, 'r') as f:
                data = json.load(f)
                data['N'] = n_value
                data['shape'] = shape
                all_trials.append(data)

df = pd.DataFrame(all_trials)

# Filter l_shape data for density sweep
df_lshape = df[df['shape'] == 'l_shape'].copy()

# Summary DataFrame for L-shape
summary_lshape = df_lshape.groupby('N').agg(
    success_rate=('success', lambda x: x.mean() * 100),
    mean_duration=('duration', 'mean'),
    mean_shuffles=('total_shuffles', 'mean'),
    mean_tortuosity=('tortuosity', 'mean')
).reset_index()

print("Data Loaded Successfully.")
print(f"Total Trials: {len(df)}")


## 1. The Phase Transition (Stochastic Shuffling to Brute Force Extrusion)
At lower densities ($N=10, 15$), the swarm relies on stochastic shuffling to generate emergent torque, maneuvering the non-convex payload. However, at $N \geq 20$, the swarm overwhelms the physical gap and transitions to *Brute Force Extrusion*, driving success to $100\%$ while shuffling collapses to zero.


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

color = '#2ecc71'
ax1.set_xlabel('Swarm Density (N)', fontsize=12)
ax1.set_ylabel('Success Rate (%)', color=color, fontsize=12)
ax1.plot(summary_lshape['N'], summary_lshape['success_rate'], color=color, marker='o', linewidth=3, markersize=10, label='Success Rate')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-5, 105)

ax2 = ax1.twinx()  
color = '#e74c3c'
ax2.set_ylabel('Mean Shuffles', color=color, fontsize=12)  
ax2.plot(summary_lshape['N'], summary_lshape['mean_shuffles'], color=color, marker='s', linestyle='--', linewidth=3, markersize=10, label='Mean Shuffles')
ax2.tick_params(axis='y', labelcolor=color)

fig.suptitle('Phase Transition: Swarm Density vs. Transport Mechanics', fontsize=16)
fig.tight_layout()  
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 2. The Torque Debt (Geometric Control)
By analyzing $N=15$ for both the `square` and `l_shape` geometries, we can isolate the "Torque Debt" imposed by non-convexity. We use a Welch's t-test to determine the statistical significance of the increased shuffling requirement.


In [ ]:
df_n15 = df[df['N'] == 15]

plt.figure(figsize=(8, 6))
sns.boxplot(x='shape', y='total_shuffles', data=df_n15, palette='pastel')
plt.title('The Torque Debt: Shuffles Required by Geometry (N=15)', fontsize=14)
plt.xlabel('Geometry Shape', fontsize=12)
plt.ylabel('Total Shuffles', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

# Welch's t-test
square_shuffles = df_n15[df_n15['shape'] == 'square']['total_shuffles']
lshape_shuffles = df_n15[df_n15['shape'] == 'l_shape']['total_shuffles']

t_stat, p_val = stats.ttest_ind(square_shuffles, lshape_shuffles, equal_var=False)
print(f"Welch's t-test Results:")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val:.4e}")

if p_val < 0.05:
    print("Conclusion: The non-convex L-shape requires a statistically significant increase in shuffling.")
else:
    print("Conclusion: No significant difference in shuffling between geometries.")


## 3. Path Complexity (Tortuosity)
As density increases, the swarm transitions from chaotic stochastic lateral movement to highly ballistic, localized extrusion.


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(summary_lshape['N'], summary_lshape['mean_tortuosity'], color='#9b59b6', marker='^', linewidth=3, markersize=10)
plt.title('Path Tortuosity vs Swarm Density (L-Shape)', fontsize=16)
plt.xlabel('Swarm Density (N)', fontsize=12)
plt.ylabel('Mean Tortuosity', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## 4. LaTeX Export
Formatted table export for the thesis document.


In [ ]:
latex_df = summary_lshape.rename(columns={
    'N': 'Swarm Size ($N$)',
    'success_rate': 'Success Rate (\%)',
    'mean_duration': 'Mean Duration (s)',
    'mean_shuffles': 'Mean Shuffles',
    'mean_tortuosity': 'Mean Tortuosity'
})

latex_out = latex_df.to_latex(index=False, float_format="%.2f", caption="Gold Standard Experimental Results: L-Shape Transport", label="tab:final_results")
print(latex_out)
